# Day 075 — Exercise 3: mux_audio_video

**What you'll build:** `mux_audio_video(video_path, audio_bytes, output_path, ffmpeg_fn=None) -> Path` — combine a silent video file with audio bytes into a playable MP4.

**Why it matters:** Stage 3 of the pipeline — this is what gives the talking-head its voice. FFmpeg's two-input mux combines the frame sequence with the TTS track.

In [ ]:
from pathlib import Path
import tempfile, subprocess
_mock_tts_fn     = lambda text, voice, rate, pitch: b'MP3:' + text[:8].encode()
_mock_mux_fn     = lambda video_path, audio_bytes, output_path: (Path(output_path).write_bytes(b'MUX' + bytes(len(audio_bytes))), Path(output_path))[1]


## Task

- **Mock:** `return ffmpeg_fn(video_path, audio_bytes, output_path)`
- **Real:** write `audio_bytes` to a `NamedTemporaryFile(.mp3)`; in `try/finally` run `subprocess.run(['ffmpeg','-y','-i',str(video_path),'-i',audio_path,'-c:v','copy','-c:a','aac','-shortest',str(out_path)], capture_output=True, text=True)`; raise `RuntimeError` if `returncode != 0`; `finally: Path(audio_path).unlink(missing_ok=True)`
- Return `Path(output_path)`

## Your Implementation

In [ ]:
def mux_audio_video(video_path, audio_bytes: bytes,
                    output_path, ffmpeg_fn=None):
    """Combine a silent video file with audio bytes into a new MP4.

    Args:
        video_path:  path to silent video file
        audio_bytes: MP3 audio bytes to add as soundtrack
        output_path: destination MP4 path
        ffmpeg_fn:   callable(video_path, audio_bytes, output_path) -> Path
    Returns:
        Path to the output video with audio
    """
    raise NotImplementedError


In [ ]:
def mux_audio_video(video_path, audio_bytes, output_path, ffmpeg_fn=None):
    if ffmpeg_fn is not None:
        return ffmpeg_fn(video_path, audio_bytes, output_path)
    with tempfile.NamedTemporaryFile(suffix='.mp3', delete=False) as f:
        f.write(audio_bytes)
        audio_path = f.name
    try:
        out_path = Path(output_path)
        result = subprocess.run(
            ['ffmpeg', '-y', '-i', str(video_path), '-i', audio_path,
             '-c:v', 'copy', '-c:a', 'aac', '-shortest', str(out_path)],
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            raise RuntimeError(f'FFmpeg mux error: {result.stderr[-500:]}')
        return out_path
    finally:
        Path(audio_path).unlink(missing_ok=True)


## Automated checks

In [ ]:

score, total = 0, 5
try:
    audio = _mock_tts_fn('Hello world', 'v', '+0%', '+0Hz')

    # create a placeholder 'silent' video file
    with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as f:
        silent = f.name
    Path(silent).write_bytes(b'SILENT_VIDEO')

    with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as f:
        out1 = f.name

    # returns Path
    result = mux_audio_video(silent, audio, out1, ffmpeg_fn=_mock_mux_fn)
    assert isinstance(result, Path), f"expected Path, got {type(result)}"
    score += 1; print("✅ returns Path")

    # file exists and non-empty
    assert result.exists() and result.stat().st_size > 0
    score += 1; print("✅ output file exists with non-zero size")

    # ffmpeg_fn receives correct arguments
    captured = {}
    def _cap_fn(vp, ab, op):
        captured.update(vp=str(vp), n_audio=len(ab))
        return Path(op)
    mux_audio_video(silent, audio, out1, ffmpeg_fn=_cap_fn)
    assert captured.get('vp') == str(silent)
    score += 1; print("✅ ffmpeg_fn receives video_path and audio_bytes")

    # audio bytes are forwarded (mock encodes length in output size)
    audio_short = _mock_tts_fn('Hi', 'v', '+0%', '+0Hz')
    audio_long  = _mock_tts_fn('Hello world!!', 'v', '+0%', '+0Hz')
    with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as f:
        out2 = f.name
    r1 = mux_audio_video(silent, audio_short, out1, ffmpeg_fn=_mock_mux_fn)
    r2 = mux_audio_video(silent, audio_long,  out2, ffmpeg_fn=_mock_mux_fn)
    assert r1.stat().st_size != r2.stat().st_size
    score += 1; print("✅ audio_bytes incorporated (different sizes)")

    # output_path is respected
    assert str(result) == out1
    score += 1; print("✅ returned Path matches output_path")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def mux_audio_video(video_path, audio_bytes, output_path, ffmpeg_fn=None):
    if ffmpeg_fn is not None:
        return ffmpeg_fn(video_path, audio_bytes, output_path)
    with tempfile.NamedTemporaryFile(suffix='.mp3', delete=False) as f:
        f.write(audio_bytes)
        audio_path = f.name
    try:
        out_path = Path(output_path)
        result = subprocess.run(
            ['ffmpeg', '-y', '-i', str(video_path), '-i', audio_path,
             '-c:v', 'copy', '-c:a', 'aac', '-shortest', str(out_path)],
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            raise RuntimeError(f'FFmpeg mux error: {result.stderr[-500:]}')
        return out_path
    finally:
        Path(audio_path).unlink(missing_ok=True)
```

**Why `finally` for cleanup?** If FFmpeg raises or returns non-zero, the temp audio file must still be deleted. `finally` runs regardless of whether the `try` block succeeded or raised.

</details>